# BÀI TẬP LỚN: QUY TRÌNH XỬ LÝ VÀ CHUẨN BỊ DỮ LIỆU
## Tập dữ liệu: Video Game Sales with Ratings (Video_Games_Sales_as_at_22_Dec_2016.csv)

**Mục tiêu:**

Để hoàn thành đề tài này, các mục tiêu cụ thể được đặt ra bao gồm:

- **Xử lý và làm sạch dữ liệu**: Sử dụng ngôn ngữ lập trình Python để kiểm tra, xử lý các giá trị thiếu (missing values) ở các cột quan trọng như năm phát hành (Year), nhà phát hành (Publisher), điểm đánh giá chuyên môn (Critic_Score) và điểm đánh giá người dùng (User_Score), đồng thời chuẩn hóa dữ liệu thô.

- **Trực quan hóa dữ liệu (EDA)**: Khám phá mối quan hệ giữa các yếu tố như thể loại game, nền tảng máy game, điểm đánh giá qua các thời kỳ và tầm ảnh hưởng của chúng đối với doanh thu tại các khu vực khác nhau (Bắc Mỹ, Châu Âu, Nhật Bản, Toàn cầu).

---

- **Tiền xử lý và chuẩn hóa đặc trưng**: Thực hiện mã hóa các biến định tính (Categorical variables) như Platform, Genre, Publisher, Rating thành dạng số để sẵn sàng đưa vào các mô hình học máy.

- **Xây dựng mô hình học máy**: Huấn luyện các mô hình dự báo (Decision Tree, Random Forest, Gradient Boosting) để dự đoán mức độ thành công (doanh thu toàn cầu) của một trò chơi, có tận dụng thêm thông tin điểm đánh giá chuyên môn/người dùng — điều mà bộ dữ liệu gốc (vgsales.csv) không có.

- **Đánh giá hiệu suất**: Sử dụng các chỉ số đo lường học máy để đánh giá độ chính xác của mô hình và đưa ra những đề xuất thực tế cho các nhà làm game.

**Lưu ý về bộ dữ liệu:** So với bộ `vgsales.csv` ban đầu, bộ dữ liệu này (`Video_Games_Sales_as_at_22_Dec_2016.csv`) có thêm 6 cột: `Critic_Score`, `Critic_Count`, `User_Score`, `User_Count`, `Developer`, `Rating`. Đây là những đặc trưng có khả năng ảnh hưởng lớn đến doanh số nhưng bị thiếu khá nhiều (khoảng 40-51% số dòng), nên cần chiến lược xử lý thiếu dữ liệu cẩn thận thay vì xóa bỏ (drop) vì sẽ mất gần một nửa dữ liệu.

## Phần 1: Data Cleansing & Điền khuyết dữ liệu gốc

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("Video_Games_Sales_as_at_22_Dec_2016.csv")

# Doi ten cot cho nhat quan voi phan con lai cua notebook
df = df.rename(columns={"Year_of_Release": "Year"})

 1.	Tổng quan về dữ liệu 

In [ ]:
display(df.head())
df.info()
display(df.describe())

 2. Thống kê dữ liệu thiếu trên các biến số và trực quan hóa dữ liệu thiếu bằng biểu đồ (Heatmap)

In [ ]:
missing_stats = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_stats / len(df) * 100).round(2)
display(pd.DataFrame({"So_luong_thieu": missing_stats, "Ty_le_thieu (%)": missing_pct}))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt # import them plt de quan ly show hinh
%matplotlib inline

plt.figure(figsize=(10, 6))
sns.heatmap(df.isna(), yticklabels=False, cbar=True, cmap='viridis')
plt.title("Ban do du lieu thieu")
plt.show()

**Nhận xét:** `Critic_Score`, `Critic_Count`, `User_Score`, `User_Count`, `Developer`, `Rating` thiếu rất nhiều (~40-51%), trong khi `Name`, `Genre`, `Year`, `Publisher` chỉ thiếu rất ít. Chiến lược xử lý:
- Với `Name`/`Genre` thiếu quá ít (2 dòng) → loại bỏ trực tiếp.
- Với `Year`/`Publisher` thiếu ít → xử lý như cách làm cũ (loại bỏ Year thiếu, điền "Unknown" cho Publisher).
- Với `Critic_Score`/`User_Score`... thiếu nhiều → **không xóa dòng** (sẽ mất gần nửa dữ liệu), thay vào đó **tạo thêm cờ đánh dấu thiếu** (`Has_Critic_Score`, `Has_User_Score`) rồi điền khuyết bằng trung vị (median) tính riêng trên tập Train ở Chương 4, để mô hình vẫn tận dụng được các dòng có review lẫn không có review.

 3. Xử lý dữ liệu trùng lặp và các dòng thiếu Name/Genre

In [ ]:
print("So dong trung lap:", df.duplicated().sum())
print("So dong thieu Name hoac Genre:", df[["Name","Genre"]].isna().any(axis=1).sum())

df = df.dropna(subset=["Name", "Genre"]).copy()
df.drop_duplicates(inplace=True)

In [ ]:
df.dtypes

 4. Xử lý Year bị thiếu

In [ ]:
missing_year = df["Year"].isna().sum()
pct_missing = missing_year / len(df) * 100
print(f"So dong thieu Year: {missing_year} ({pct_missing:.2f}% tong du lieu)")

df = df.dropna(subset=["Year"])
df["Year"] = df["Year"].astype(int)

 5. Xử lý Publisher

In [ ]:
df["Publisher"] = df["Publisher"].fillna("Unknown")

 6. Xử lý Critic_Score, Critic_Count, User_Score, User_Count, Developer, Rating

- `User_Score` đang ở dạng chuỗi (object) vì có giá trị `"tbd"` (to be determined) lẫn trong các số — cần ép kiểu numeric, `"tbd"` sẽ tự động thành NaN.
- Tạo cờ `Has_Critic_Score`, `Has_User_Score` để mô hình biết được dòng nào có/không có đánh giá (bản thân việc "có review hay không" cũng có thể mang thông tin — game review nhiều thường là game được chú ý).
- `Critic_Score`/`Critic_Count`/`User_Score`/`User_Count` **chưa điền khuyết ở đây** — việc điền trung vị sẽ thực hiện ở mục 4.5 sau khi chia Train/Test, để tránh rò rỉ dữ liệu (giống nguyên tắc áp dụng với Target Encoding).
- `Developer`, `Rating` là biến định tính → điền `"Unknown"`.

In [ ]:
# Ep kieu User_Score ve numeric ("tbd" -> NaN tu dong)
df["User_Score"] = pd.to_numeric(df["User_Score"], errors="coerce")

# Dua User_Score ve cung thang diem voi Critic_Score (0-100) de de so sanh/truc quan
df["User_Score_100"] = df["User_Score"] * 10

# Co danh dau co/khong co danh gia
df["Has_Critic_Score"] = df["Critic_Score"].notna().astype(int)
df["Has_User_Score"] = df["User_Score"].notna().astype(int)

# Bien dinh tinh thieu -> Unknown
df["Developer"] = df["Developer"].fillna("Unknown")
df["Rating"] = df["Rating"].fillna("Unknown")

print("Ty le co Critic_Score:", df["Has_Critic_Score"].mean().round(3))
print("Ty le co User_Score  :", df["Has_User_Score"].mean().round(3))

 7. Tạo bảng năm ra mắt nền tảng (Platform Launch), thế hệ máy của máy và gộp vào dữ liệu chính 

In [ ]:
platform_launch = {
    "Platform": [
        "2600","3DO","3DS","DC","DS","GB","GBA","GC","GEN","GG",
        "N64","NES","NG","PC","PCFX","PS","PS2","PS3","PS4","PSP",
        "PSV","SAT","SCD","SNES","TG16","WS","Wii","WiiU","X360",
        "XB","XOne"
    ],

    "Platform_Launch_Year":[
        1977,1993,2011,1998,2004,1989,2001,2001,1988,1990,
        1996,1983,1990,1981,1994,1994,2000,2006,2013,2004,
        2011,1994,1991,1990,1987,1999,2006,2012,2005,
        2001,2013
    ]
}

df_launch = pd.DataFrame(platform_launch)

In [ ]:
df = df.merge(df_launch,
              on="Platform",
              how="left")

In [ ]:
generation = {
    "Platform": [
        "2600", "NES", "GB", "GEN", "GG", "SNES", "SCD", "TG16", "3DO", "PCFX",
        "PS", "SAT", "N64", "DC", "PS2", "GC", "GBA", "XB", "WS", "X360",
        "Wii", "PS3", "PSP", "3DS", "WiiU", "PS4", "XOne", "PSV"
    ],
    "Generation": [
        2, 3, 4, 4, 4, 4, 4, 4, 5, 5,
        5, 5, 5, 6, 6, 6, 6, 6, 6, 7,
        7, 7, 7, 8, 8, 8, 8, 8
    ]
}

df_generation = pd.DataFrame(generation)

In [ ]:
df = df.merge(
    df_generation,
    on="Platform",
    how="left"
)
df["Generation"] = df["Generation"].fillna(0)

8. Trích xuất Franchise (thương hiệu) và cờ Sequel từ tên game

In [ ]:
def extract_franchise(name):
    """Cat phan phu de (sau dau ':' hoac '-') va so/so La Ma o cuoi de lay ten thuong hieu goc."""
    if pd.isna(name):
        return "Unknown"
    s = str(name)
    for sep in [":", " - ", "\u2013"]:
        if sep in s:
            s = s.split(sep)[0]
            break
    s = re.sub(r"\s+(I|II|III|IV|V|VI|VII|VIII|IX|X|\d+)$", "", s.strip())
    s = re.sub(r"\s+(Remastered|Deluxe|Edition|HD|Ultimate|Complete|Collection)$", "", s.strip(), flags=re.IGNORECASE)
    return s.strip()

def is_sequel(name):
    """Danh dau 1 neu ten game ket thuc bang so La Ma hoac chu so >= 2 (heuristic don gian)."""
    if pd.isna(name):
        return 0
    s = str(name).strip()
    return int(bool(re.search(r"\b(II|III|IV|V|VI|VII|VIII|IX|X|[2-9])$", s)))

df["Franchise"] = df["Name"].apply(extract_franchise)
df["Is_Sequel"] = df["Name"].apply(is_sequel)

print(f"So franchise duy nhat: {df['Franchise'].nunique()}")
print(f"So game duoc danh dau la sequel: {df['Is_Sequel'].sum()} ({df['Is_Sequel'].mean():.1%})")

9. Tạo thêm feature Years_After_Launch và chia vòng đời console (Life Cycle)

In [ ]:
df["Years_After_Launch"] = (
    df["Year"] -
    df["Platform_Launch_Year"]
)

In [ ]:
n_negative = (df["Years_After_Launch"] < 0).sum()
n_missing = df["Years_After_Launch"].isna().sum()
print(f"So dong Years_After_Launch am (bat thuong): {n_negative}")
print(f"So dong Years_After_Launch NaN (do thieu Platform_Launch_Year): {n_missing}")

df = df[df["Years_After_Launch"] >= 0]

In [ ]:
def life_cycle(x):
    if x <= 2:
        return "Launch"
    elif x <= 5:
        return "Growth"
    elif x <= 8:
        return "Mature"
    else:
        return "Late"

df["Life_Cycle"] = df["Years_After_Launch"].apply(life_cycle)

10. Kiểm tra dữ liệu sau khi thêm feature

In [ ]:
df.describe(include="all")

11. Kiểm tra Outlier ở các cột doanh số

In [ ]:
sales = ["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"]

plt.figure(figsize=(10,5))
sns.boxplot(data=df[sales])
plt.title("Sales Distribution")
plt.show()

12. Kiểm tra Global_Sales tính toán lại có khớp không

In [ ]:
df["Calculated_Global_Sales"] = (
      df["NA_Sales"]
    + df["EU_Sales"]
    + df["JP_Sales"]
    + df["Other_Sales"]
)

print((df["Calculated_Global_Sales"] == df["Global_Sales"]).value_counts())

13. Tổng quan lại dữ liệu sau khi làm sạch (các biến định danh/định tính)

Ở bước này, `Name`, `Genre`, `Year`, `Publisher`, `Developer`, `Rating` đã hoàn toàn sạch (không còn thiếu). Riêng `Critic_Score`, `Critic_Count`, `User_Score`, `User_Count`, `User_Score_100` **vẫn còn thiếu có chủ đích** — việc điền khuyết các cột này bị dời xuống mục 4.5 (sau khi chia Train/Test), vì điền ngay bây giờ bằng thống kê của toàn bộ dữ liệu (bao gồm cả phần sẽ thành tập Test) sẽ gây rò rỉ dữ liệu, giống nguyên tắc áp dụng với Target Encoding.

In [ ]:
df.info()
display(df.describe(include="all"))

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isna(), yticklabels=False, cbar=True, cmap='viridis')
plt.title("Ban do du lieu thieu (Critic/User Score con thieu co chu dich, se dien o Chuong 4)")
plt.show()

print("Cac cot da sach hoan toan:", [c for c in df.columns if df[c].isna().sum() == 0])
print("\nCac cot con thieu co chu dich (se dien khuyet o muc 4.5, sau Train/Test split):")
still_missing = df.isna().sum()
still_missing = still_missing[still_missing > 0]
print(still_missing)

# PHẦN 2: Đưa dữ liệu vào cơ sở dữ liệu MySQL

1. Xuất dữ liệu mới được làm sạch sang file mới

In [ ]:
print("Xuat file CSV")
df.to_csv("Cleaned_vgsales_ratings.csv", index=False, encoding="utf-8")

2. Đọc lại file mới 

In [ ]:
import pandas as pd
df = pd.read_csv("Cleaned_vgsales_ratings.csv")

3. Kết nối với server MySQL

In [ ]:
%pip install sqlalchemy mysql-connector-python

In [ ]:
# Ket noi MySQL
import os
from sqlalchemy import create_engine

# Uu tien lay tu bien moi truong; neu chua set thi dung gia tri mac dinh cho moi truong local
username = os.getenv("DB_USER", "root")
password = os.getenv("DB_PASSWORD", "12345")  # KHUYEN NGHI: dat DB_PASSWORD qua bien moi truong khi dung o moi truong that
host = os.getenv("DB_HOST", "localhost")
database = os.getenv("DB_NAME", "data_cleaned1")

engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}/{database}"
)

4. Ghi vào server 

In [ ]:
# Ghi vao MySQL
df.to_sql(
    name="vgsales_ratings_cleaned",
    con=engine,
    if_exists="replace",
    index=False
)

print("Luu thanh cong!")

# PHẦN 3: KHAI THÁC THÔNG TIN HỮU ÍCH – EDA 

1. Doanh thu Game thay đổi như thế nào trong vòng đời của một hệ máy?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.boxplot(
    data=df,
    x="Life_Cycle",
    y="Global_Sales",
    order=["Launch", "Growth", "Mature", "Late"],
    showfliers=False
)
plt.show()

2. Thể loại game thay đổi như thế nào theo thời gian?

In [ ]:
genre_year = (
    df.groupby(["Year","Genre"])
      .size()
      .reset_index(name="Count")
)

plt.figure(figsize=(10,5))
sns.lineplot(
    data=genre_year,
    x="Year",
    y="Count",
    hue="Genre"
)
plt.show()

3. Doanh thu dịch chuyển giữa các khu vực như thế nào?

In [ ]:
sales_by_year = df.groupby("Year")[
    ["NA_Sales","EU_Sales","JP_Sales","Other_Sales"]
].sum()

sales_by_year.plot.area(figsize=(10,5))
plt.show()

4. Console thế hệ nào tạo ra nhiều doanh thu nhất?

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=df,
    x="Generation",
    y="Global_Sales",
    estimator=sum
)
plt.show()

5. Sau bao nhiêu năm kể từ khi console ra mắt thì doanh số game đạt đỉnh?

In [ ]:
year_after = (
    df.groupby("Years_After_Launch")["Global_Sales"].sum()
)
year_after.plot(figsize=(10,5))
plt.show()

6. Mức độ thành công của các nhà phát hành (Publisher) qua các giai đoạn

In [ ]:
top_pub = (
    df.groupby("Publisher")["Global_Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

plt.figure(figsize=(10,6))
sns.barplot(x=top_pub.values, y=top_pub.index)
plt.title("Top 10 Publishers by Global Sales")
plt.xlabel("Global Sales (Million Copies)")
plt.ylabel("Publisher")
plt.show()

7. Doanh số game có đang giảm sau năm 2008?

In [ ]:
sales_year = df.groupby("Year")["Global_Sales"].sum()
sales_year.plot(figsize=(10,5))
plt.show()

8. Điểm đánh giá chuyên môn (Critic_Score) và người dùng (User_Score) có liên quan đến doanh số không?

Đây là phân tích quan trọng nhất với bộ dữ liệu mới — kiểm tra xem việc bổ sung điểm đánh giá có thực sự đáng làm hay không.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df[df["Has_Critic_Score"]==1], x="Critic_Score", y="Global_Sales", alpha=0.4, ax=axes[0])
axes[0].set_title("Critic Score vs Global Sales")

sns.scatterplot(data=df[df["Has_User_Score"]==1], x="User_Score_100", y="Global_Sales", alpha=0.4, ax=axes[1])
axes[1].set_title("User Score (thang 100) vs Global Sales")

plt.tight_layout()
plt.show()

print("He so tuong quan Critic_Score - Global_Sales:",
      round(df.loc[df["Has_Critic_Score"]==1, ["Critic_Score","Global_Sales"]].corr().iloc[0,1], 3))
print("He so tuong quan User_Score  - Global_Sales:",
      round(df.loc[df["Has_User_Score"]==1, ["User_Score_100","Global_Sales"]].corr().iloc[0,1], 3))

9. Rating (ESRB) ảnh hưởng thế nào đến doanh số trung bình?

In [ ]:
rating_sales = (
    df[df["Rating"] != "Unknown"]
    .groupby("Rating")["Global_Sales"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8,5))
sns.barplot(x=rating_sales.values, y=rating_sales.index)
plt.title("Doanh so trung binh theo ESRB Rating")
plt.xlabel("Global Sales trung binh (trieu ban)")
plt.show()

10. Mỗi nền tảng (Platform) đạt doanh số đỉnh sau bao nhiêu năm?

In [ ]:
peak = (
    df.groupby(["Platform","Years_After_Launch"])["Global_Sales"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(12,6))
sns.lineplot(data=peak, x="Years_After_Launch", y="Global_Sales", hue="Platform")
plt.show()

# CHƯƠNG 4: PHÂN TÍCH DỮ LIỆU ĐƯA VÀO MÔ HÌNH MÁY HỌC

## 4.1 Lựa chọn biến đầu vào và loại bỏ outlier cực hạn

Loại bỏ các bản ghi có `Global_Sales` vượt phân vị 99.9% (các siêu phẩm cá biệt như *Wii Sports*, *GTA V*...) để mô hình học được xu hướng chung trên phần lớn dữ liệu, tránh bị nhóm nhỏ giá trị cực đoan này chi phối quá trình huấn luyện.

**Lưu ý quan trọng:** `Publisher_Mean_Sales`, `Genre_Platform_Quality`, `Franchise_Mean_Sales`, `Developer_Mean_Sales` **chỉ được khai báo tên** ở đây. Việc tính toán giá trị cụ thể (Target Encoding có Bayesian Smoothing) sẽ được thực hiện ở mục **4.5**, sau khi đã chia Train/Test, nhằm tránh rò rỉ dữ liệu. Tương tự, việc điền khuyết `Critic_Score`/`Critic_Count`/`User_Score`/`User_Count` (bằng trung vị) cũng chỉ thực hiện sau khi chia Train/Test ở mục 4.5.

**Đặc trưng mới từ bộ dữ liệu giàu thông tin hơn:** `Critic_Score`, `Critic_Count`, `User_Score_100`, `User_Count`, `Has_Critic_Score`, `Has_User_Score`, `Rating`, `Developer_Mean_Sales` — đây là các đặc trưng kỳ vọng cải thiện đáng kể so với bộ `vgsales.csv` gốc.

In [ ]:
outlier_threshold = df["Global_Sales"].quantile(0.999)
print(f"Nguong loc theo phan vi 99.9%: {outlier_threshold:.2f} trieu ban")

df_filtered = df[df["Global_Sales"] <= outlier_threshold].copy()

selected_features = [
    "Platform",
    "Genre",
    "Rating",
    "Publisher_Mean_Sales",
    "Genre_Platform_Quality",
    "Franchise_Mean_Sales",
    "Developer_Mean_Sales",
    "Is_Sequel",
    "Critic_Score",
    "Critic_Count",
    "User_Score_100",
    "User_Count",
    "Has_Critic_Score",
    "Has_User_Score",
    "Year",
    "Platform_Launch_Year",
    "Years_After_Launch",
    "Life_Cycle",
    "Generation",
]
target = "Global_Sales"

print(f"So ban ghi truoc khi loc: {df.shape[0]}")
print(f"So ban ghi sau khi loc  : {df_filtered.shape[0]} ({df_filtered.shape[0]/df.shape[0]:.1%})")

## 4.2 Phân tích phân phối biến mục tiêu (Global_Sales)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_filtered["Global_Sales"], bins=40, kde=True)
plt.title("Distribution of Global Sales")
plt.xlabel("Global Sales")
plt.ylabel("Frequency")
plt.show()

print("Do lech (skewness):", df_filtered["Global_Sales"].skew().round(2))
print("=> Phan phoi lech phai manh, can dung log1p(Global_Sales) lam bien muc tieu khi huan luyen (xem muc 4.5).")

## 4.3 Phân tích Outlier còn lại và tương quan giữa các biến số

In [ ]:
numeric_df = df_filtered[[
    "Year", "Platform_Launch_Year", "Years_After_Launch", "Generation",
    "Critic_Score", "User_Score_100", "Global_Sales"
]]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(y=df_filtered["Global_Sales"], ax=axes[0])
axes[0].set_title("Boxplot of Global Sales (sau khi loc)")

sns.heatmap(numeric_df.corr(), annot=True, cmap="YlGnBu", fmt=".2f", ax=axes[1])
axes[1].set_title("Correlation Matrix")

plt.tight_layout()
plt.show()

## 4.4 Phân tích mối quan hệ giữa đặc trưng và doanh số

In [ ]:
genre_sales = (
    df_filtered.groupby("Genre")["Global_Sales"]
      .mean()
      .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(x=genre_sales.values, y=genre_sales.index, ax=axes[0])
axes[0].set_title("Average Global Sales by Genre")
axes[0].set_xlabel("Global Sales")

sns.scatterplot(data=df_filtered, x="Years_After_Launch", y="Global_Sales", alpha=0.5, ax=axes[1])
axes[1].set_title("Years After Launch vs Global Sales")

plt.tight_layout()
plt.show()

## 4.5 Chia tập Train/Test, điền khuyết Critic/User Score và xây dựng đặc trưng Target Encoding (khắc phục rò rỉ dữ liệu)

**Nguyên tắc:** mọi thống kê tính từ dữ liệu (trung bình theo nhóm, trung vị điền khuyết...) đều phải chỉ học từ tập Train, sau đó ánh xạ (map) sang tập Test — tránh trường hợp thông tin của tập kiểm thử "rò rỉ" vào bước chuẩn bị đặc trưng.

1. Chia Train/Test **trước tiên**, dựa trên chỉ số dòng của `df_filtered`.
2. Điền khuyết `Critic_Score`, `Critic_Count`, `User_Score_100`, `User_Count` bằng **trung vị của tập Train**.
3. Tính Target Encoding có **Bayesian Smoothing** cho `Publisher`, cặp `(Genre, Platform)`, `Franchise`, `Developer` — chỉ trên tập Train, ánh xạ sang Test (nhóm chưa từng gặp → dùng trung bình toàn cục của Train).

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(df_filtered.index, test_size=0.2, random_state=42)
train_df = df_filtered.loc[train_idx].copy()
test_df = df_filtered.loc[test_idx].copy()

global_mean_sales = train_df["Global_Sales"].mean()
SMOOTHING_K = 10  # he so lam muot: nhom cang it mau thi cang bi keo ve trung binh toan cuc

# ---- 1. Dien khuyet Critic/User Score bang trung vi cua Train ----
impute_cols = ["Critic_Score", "Critic_Count", "User_Score_100", "User_Count"]
impute_medians = train_df[impute_cols].median()

for col in impute_cols:
    train_df[col] = train_df[col].fillna(impute_medians[col])
    test_df[col] = test_df[col].fillna(impute_medians[col])

print("Trung vi dung de dien khuyet (tinh tren Train):")
print(impute_medians)

# ---- 2. Ham Target Encoding co Bayesian Smoothing ----
def smoothed_target_encode(train_series_group, train_target, global_mean, k=SMOOTHING_K):
    """Tra ve map {group_key: smoothed_mean} de tranh overfit voi nhom co it mau."""
    stats = train_target.groupby(train_series_group).agg(["mean", "count"])
    smoothed = (stats["count"] * stats["mean"] + k * global_mean) / (stats["count"] + k)
    return smoothed

# Publisher_Mean_Sales
publisher_mean_map = smoothed_target_encode(train_df["Publisher"], train_df["Global_Sales"], global_mean_sales)
train_df["Publisher_Mean_Sales"] = train_df["Publisher"].map(publisher_mean_map)
test_df["Publisher_Mean_Sales"] = test_df["Publisher"].map(publisher_mean_map).fillna(global_mean_sales)

# Genre_Platform_Quality
gp_key_train = list(zip(train_df["Genre"], train_df["Platform"]))
gp_mean_map = smoothed_target_encode(pd.Series(gp_key_train, index=train_df.index), train_df["Global_Sales"], global_mean_sales)
train_df["Genre_Platform_Quality"] = train_df.set_index(["Genre", "Platform"]).index.map(gp_mean_map)
test_df["Genre_Platform_Quality"] = test_df.set_index(["Genre", "Platform"]).index.map(gp_mean_map)
test_df["Genre_Platform_Quality"] = test_df["Genre_Platform_Quality"].fillna(global_mean_sales)

# Franchise_Mean_Sales (k lon hon vi nhieu franchise chi co 1 game)
franchise_mean_map = smoothed_target_encode(train_df["Franchise"], train_df["Global_Sales"], global_mean_sales, k=30)
train_df["Franchise_Mean_Sales"] = train_df["Franchise"].map(franchise_mean_map)
test_df["Franchise_Mean_Sales"] = test_df["Franchise"].map(franchise_mean_map).fillna(global_mean_sales)

# Developer_Mean_Sales (k lon vi rat nhieu gia tri thieu da duoc dien "Unknown")
developer_mean_map = smoothed_target_encode(train_df["Developer"], train_df["Global_Sales"], global_mean_sales, k=20)
train_df["Developer_Mean_Sales"] = train_df["Developer"].map(developer_mean_map)
test_df["Developer_Mean_Sales"] = test_df["Developer"].map(developer_mean_map).fillna(global_mean_sales)

# Bien muc tieu: log1p de giam do lech phan phoi (xem muc 4.2)
y_train = np.log1p(train_df[target])
y_test = np.log1p(test_df[target])

print("\nTrain:", train_df.shape, " Test:", test_df.shape)
print("So gia tri thieu o dac trung da chon (Train):", train_df[selected_features].isna().sum().sum())
print("So gia tri thieu o dac trung da chon (Test) :", test_df[selected_features].isna().sum().sum())

## 4.6 Mã hóa dữ liệu (One-Hot Encoding)

Việc `fit` One-Hot Encoding cũng cần thực hiện **trên tập Train**, sau đó `reindex` tập Test theo đúng danh sách cột của Train. `Rating` (ESRB) và `Life_Cycle` cũng là biến định tính nên được one-hot cùng `Platform`, `Genre`.

In [ ]:
X_train_raw = train_df[selected_features]
X_test_raw = test_df[selected_features]

X_train_encoded = pd.get_dummies(X_train_raw, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)

# Dong bo cot giua Test va Train (tranh lech cot do khac biet category)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Kich thuoc Train sau Encoding:", X_train_encoded.shape)
print("Kich thuoc Test  sau Encoding:", X_test_encoded.shape)
display(X_train_encoded.head())

### 4.7 Xây dựng mô hình để đưa vào máy học

Sau khi dữ liệu đã được tiền xử lý, mã hóa và chia thành tập huấn luyện và kiểm thử, tiến hành xây dựng hai mô hình hồi quy nhằm dự đoán doanh số toàn cầu (`Global_Sales`) của trò chơi điện tử. Hai mô hình được lựa chọn là **Decision Tree Regressor** và **Random Forest Regressor**.

#### 4.7.1 Lý do lựa chọn mô hình

##### 1. Decision Tree Regressor
**Decision Tree Regressor** là mô hình hồi quy dựa trên cấu trúc cây quyết định. Mô hình phân chia dữ liệu thành nhiều nhánh dựa trên các điều kiện của thuộc tính nhằm giảm sai số dự đoán tại mỗi nút.

*Mô hình được lựa chọn vì:*
- Có khả năng xử lý đồng thời dữ liệu số và dữ liệu đã mã hóa *One-Hot Encoding*.
- Dễ diễn giải quá trình ra quyết định (tính trực quan cao).
- Không yêu cầu dữ liệu tuân theo phân phối chuẩn.
- Phù hợp làm mô hình cơ sở (*Baseline model*) để so sánh với các mô hình mạnh hơn.

##### 2. Random Forest Regressor
**Random Forest Regressor** là mô hình tập hợp (*Ensemble Learning*), được xây dựng từ nhiều cây quyết định khác nhau. Mỗi cây được huấn luyện trên một tập dữ liệu bootstrap và chỉ sử dụng một phần đặc trưng tại mỗi lần phân chia.

*Ưu điểm nổi bật:*
- Giảm thiểu hiện tượng *Overfitting* (quá khớp) của Decision Tree.
- Đưa ra dự đoán ổn định và có độ chính xác cao hơn.
- Khả năng tổng quát hóa tốt trên dữ liệu mới.
- Thường đạt hiệu suất và độ chính xác rất cao đối với dữ liệu dạng bảng (*Tabular Data*).

4.7.2 Huấn luyện mô hình Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

dt_model.fit(X_train_encoded, y_train)

print("Huan luyen Decision Tree hoan thanh!")

4.7.3 Huấn luyện mô hình Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_encoded, y_train)

print("Huan luyen Random Forest hoan thanh!")

4.7.4 Đánh giá độ ổn định bằng Cross-Validation (K-Fold)

Chỉ đánh giá qua **một lần** chia Train/Test có thể khiến kết quả phụ thuộc vào cách chia dữ liệu ngẫu nhiên đó. Áp dụng thêm **5-Fold Cross-Validation trên tập Train** để kiểm tra mô hình có ổn định giữa các lần chia dữ liệu khác nhau hay không.

In [ ]:
from sklearn.model_selection import cross_val_score

dt_cv_r2 = cross_val_score(dt_model, X_train_encoded, y_train, cv=5, scoring="r2")
rf_cv_r2 = cross_val_score(rf_model, X_train_encoded, y_train, cv=5, scoring="r2")

print("========== 5-Fold Cross-Validation (R2) ==========")
print(f"Decision Tree : {dt_cv_r2.mean():.4f} (+/- {dt_cv_r2.std():.4f})")
print(f"Random Forest : {rf_cv_r2.mean():.4f} (+/- {rf_cv_r2.std():.4f})")

## 4.8 Hoàn tất quá trình xây dựng mô hình cơ sở

In [ ]:
print("=" * 50)
print("HOAN THANH XAY DUNG MO HINH CO SO")
print("=" * 50)

print(f"So mau huan luyen        : {X_train_encoded.shape[0]}")
print(f"So mau kiem thu           : {X_test_encoded.shape[0]}")
print(f"So dac trung sau Encoding : {X_train_encoded.shape[1]}")

print("\nCac mo hinh da duoc huan luyen:")
print("- Decision Tree Regressor")
print("- Random Forest Regressor")

## 4.9 Tối ưu siêu tham số bằng RandomizedSearchCV

Thay vì chọn `max_depth`, `n_estimators`... thủ công như mục 4.7, sử dụng `RandomizedSearchCV` để tìm bộ siêu tham số tốt hơn cho Random Forest, dựa trên 5-Fold Cross-Validation **trên tập Train** (không đụng đến tập Test).

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [6, 8, 10, 12, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1,
)

rf_search.fit(X_train_encoded, y_train)

print("Bo tham so tot nhat:", rf_search.best_params_)
print("R2 CV tot nhat     :", round(rf_search.best_score_, 4))

rf_model_tuned = rf_search.best_estimator_

## 4.10 Thử nghiệm thêm Gradient Boosting Regressor

Ngoài Decision Tree và Random Forest, thử thêm **Gradient Boosting Regressor** — mô hình ensemble xây tuần tự (mỗi cây sau sửa lỗi của các cây trước), thường cho kết quả tốt trên dữ liệu dạng bảng (tabular). Cũng dùng `RandomizedSearchCV` để tìm tham số tốt trên tập Train.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_param_dist = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5],
    "subsample": [0.8, 0.9, 1.0],
}

gb_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=gb_param_dist,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1,
)

gb_search.fit(X_train_encoded, y_train)

print("Bo tham so tot nhat (Gradient Boosting):", gb_search.best_params_)
print("R2 CV tot nhat                        :", round(gb_search.best_score_, 4))

gb_model = gb_search.best_estimator_

# CHƯƠNG 5: ĐÁNH GIÁ VÀ SO SÁNH CÁC MÔ HÌNH HỌC MÁY

## 5.1 Dự đoán trên tập kiểm thử (Test set)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Du doan (thang log)
dt_pred_log = dt_model.predict(X_test_encoded)
rf_pred_log = rf_model.predict(X_test_encoded)
rf_tuned_pred_log = rf_model_tuned.predict(X_test_encoded)
gb_pred_log = gb_model.predict(X_test_encoded)

# Quy doi ve thang do goc (trieu ban)
dt_pred = np.expm1(dt_pred_log)
rf_pred = np.expm1(rf_pred_log)
rf_tuned_pred = np.expm1(rf_tuned_pred_log)
gb_pred = np.expm1(gb_pred_log)
y_test_orig = np.expm1(y_test)

# Baseline ngay tho: du doan bang trung binh cua tap Train (thang goc)
baseline_pred = np.full_like(y_test_orig, fill_value=np.expm1(y_train).mean(), dtype=float)

## 5.2 Tính toán các chỉ số đánh giá (MAE, RMSE, R2)

In [ ]:
def get_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

dt_mae_log, dt_rmse_log, dt_r2_log = get_metrics(y_test, dt_pred_log)
rf_mae_log, rf_rmse_log, rf_r2_log = get_metrics(y_test, rf_pred_log)
rf_tuned_mae_log, rf_tuned_rmse_log, rf_tuned_r2_log = get_metrics(y_test, rf_tuned_pred_log)
gb_mae_log, gb_rmse_log, gb_r2_log = get_metrics(y_test, gb_pred_log)

dt_mae, dt_rmse, dt_r2 = get_metrics(y_test_orig, dt_pred)
rf_mae, rf_rmse, rf_r2 = get_metrics(y_test_orig, rf_pred)
rf_tuned_mae, rf_tuned_rmse, rf_tuned_r2 = get_metrics(y_test_orig, rf_tuned_pred)
gb_mae, gb_rmse, gb_r2 = get_metrics(y_test_orig, gb_pred)
baseline_mae, baseline_rmse, baseline_r2 = get_metrics(y_test_orig, baseline_pred)

print("========== Baseline (du doan bang trung binh Train) ==========")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")
print(f"R2   : {baseline_r2:.4f}")

print("\n========== Decision Tree (thang goc - trieu ban) ==========")
print(f"MAE  : {dt_mae:.4f}")
print(f"RMSE : {dt_rmse:.4f}")
print(f"R2   : {dt_r2:.4f}")

print("\n========== Random Forest (thang goc - trieu ban) ==========")
print(f"MAE  : {rf_mae:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R2   : {rf_r2:.4f}")

print("\n========== Random Forest (da toi uu tham so, thang goc - trieu ban) ==========")
print(f"MAE  : {rf_tuned_mae:.4f}")
print(f"RMSE : {rf_tuned_rmse:.4f}")
print(f"R2   : {rf_tuned_r2:.4f}")

print("\n========== Gradient Boosting (thang goc - trieu ban) ==========")
print(f"MAE  : {gb_mae:.4f}")
print(f"RMSE : {gb_rmse:.4f}")
print(f"R2   : {gb_r2:.4f}")

## 5.3 Bảng tổng hợp kết quả so sánh

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline (mean)", "Decision Tree", "Random Forest", "Random Forest (Tuned)", "Gradient Boosting"],
    "MAE (log)": [np.nan, dt_mae_log, rf_mae_log, rf_tuned_mae_log, gb_mae_log],
    "RMSE (log)": [np.nan, dt_rmse_log, rf_rmse_log, rf_tuned_rmse_log, gb_rmse_log],
    "R2 (log)": [np.nan, dt_r2_log, rf_r2_log, rf_tuned_r2_log, gb_r2_log],
    "MAE (trieu ban)": [baseline_mae, dt_mae, rf_mae, rf_tuned_mae, gb_mae],
    "RMSE (trieu ban)": [baseline_rmse, dt_rmse, rf_rmse, rf_tuned_rmse, gb_rmse],
    "R2 (goc)": [baseline_r2, dt_r2, rf_r2, rf_tuned_r2, gb_r2],
})

display(results)

## 5.4 Trực quan hóa so sánh các mô hình

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=results, x="Model", y="MAE (trieu ban)", ax=axes[0])
axes[0].set_title("So sanh MAE")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=results, x="Model", y="RMSE (trieu ban)", ax=axes[1])
axes[1].set_title("So sanh RMSE")
axes[1].tick_params(axis="x", rotation=30)

sns.barplot(data=results, x="Model", y="R2 (goc)", ax=axes[2])
axes[2].set_title("So sanh R2 Score")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 5.4b Xác định mô hình tốt nhất (dùng chung cho các phần phân tích bên dưới)

Từ đây, các phần phân tích sâu (Feature Importance, Residual, lưu model) sẽ dùng **mô hình tốt nhất** trong 4 mô hình đã huấn luyện (không tính Baseline), dựa trên R² (thang gốc).

In [ ]:
model_candidates = {
    "Decision Tree": (dt_model, dt_r2, dt_pred, dt_pred_log),
    "Random Forest": (rf_model, rf_r2, rf_pred, rf_pred_log),
    "Random Forest (Tuned)": (rf_model_tuned, rf_tuned_r2, rf_tuned_pred, rf_tuned_pred_log),
    "Gradient Boosting": (gb_model, gb_r2, gb_pred, gb_pred_log),
}

best_model_name = max(model_candidates, key=lambda name: model_candidates[name][1])
best_model, best_r2, best_pred, best_pred_log = model_candidates[best_model_name]

print(f"Mo hinh tot nhat: {best_model_name} (R2 = {best_r2:.4f})")

## 5.5 Feature Importance – So sánh mức độ ảnh hưởng của các đặc trưng

In [ ]:
dt_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": dt_model.feature_importances_
}).sort_values("Importance", ascending=False).head(10)

rf_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False).head(10)

rf_tuned_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": rf_model_tuned.feature_importances_
}).sort_values("Importance", ascending=False).head(10)

gb_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": gb_model.feature_importances_
}).sort_values("Importance", ascending=False).head(10)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

sns.barplot(data=dt_importance, x="Importance", y="Feature", ax=axes[0])
axes[0].set_title("Decision Tree")

sns.barplot(data=rf_importance, x="Importance", y="Feature", ax=axes[1])
axes[1].set_title("Random Forest")

sns.barplot(data=rf_tuned_importance, x="Importance", y="Feature", ax=axes[2])
axes[2].set_title("Random Forest (Tuned)")

sns.barplot(data=gb_importance, x="Importance", y="Feature", ax=axes[3])
axes[3].set_title("Gradient Boosting")

plt.suptitle("Top 10 Feature Importance - So sanh 4 mo hinh", y=1.03)
plt.tight_layout()
plt.show()

## 5.6 Phân tích dự đoán thực tế và phần dư (Residuals) – Mô hình tốt nhất

Ngoài biểu đồ Actual vs Predicted, bổ sung thêm biểu đồ phần dư (residual) để kiểm tra mô hình có bị lệch (bias) hoặc phương sai không đồng đều (heteroscedasticity) ở các vùng doanh số khác nhau hay không. Sử dụng `best_model` (mô hình có R² cao nhất, xác định ở mục 5.4b).

In [ ]:
residuals = y_test_orig - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_orig, best_pred, alpha=0.5)
axes[0].plot(
    [y_test_orig.min(), y_test_orig.max()],
    [y_test_orig.min(), y_test_orig.max()],
    "r--"
)
axes[0].set_xlabel("Actual Global Sales")
axes[0].set_ylabel("Predicted Global Sales")
axes[0].set_title(f"Actual vs Predicted ({best_model_name})")

axes[1].scatter(best_pred, residuals, alpha=0.5)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted Global Sales")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].set_title(f"Residual Plot ({best_model_name})")

plt.tight_layout()
plt.show()

## 5.6b Phân tích lỗi theo phân khúc (Segment Error Analysis)

Chỉ nhìn MAE/RMSE/R² tổng thể có thể che khuất việc mô hình dự đoán rất tệ ở một số nhóm cụ thể. Phân tích lỗi của `best_model` theo từng **Genre** và từng **Platform** để biết mô hình yếu ở phân khúc nào.

In [ ]:
test_df_eval = test_df.copy()
test_df_eval["Actual"] = y_test_orig.values
test_df_eval["Predicted"] = best_pred
test_df_eval["Abs_Error"] = (test_df_eval["Actual"] - test_df_eval["Predicted"]).abs()

# --- Loi theo Genre ---
genre_error = (
    test_df_eval.groupby("Genre")
    .apply(lambda g: pd.Series({
        "MAE": g["Abs_Error"].mean(),
        "RMSE": np.sqrt((g["Abs_Error"] ** 2).mean()),
        "So_luong_mau": len(g),
    }))
    .sort_values("MAE", ascending=False)
)

# --- Loi theo Platform (chi lay Platform co >= 20 mau trong Test de tranh nhieu) ---
platform_error = (
    test_df_eval.groupby("Platform")
    .apply(lambda g: pd.Series({
        "MAE": g["Abs_Error"].mean(),
        "RMSE": np.sqrt((g["Abs_Error"] ** 2).mean()),
        "So_luong_mau": len(g),
    }))
)
platform_error = platform_error[platform_error["So_luong_mau"] >= 20].sort_values("MAE", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=genre_error["MAE"], y=genre_error.index, ax=axes[0])
axes[0].set_title(f"MAE theo Genre ({best_model_name})")
axes[0].set_xlabel("MAE (trieu ban)")

sns.barplot(x=platform_error["MAE"], y=platform_error.index, ax=axes[1])
axes[1].set_title(f"MAE theo Platform, chi Platform co >= 20 mau ({best_model_name})")
axes[1].set_xlabel("MAE (trieu ban)")

plt.tight_layout()
plt.show()

print("========== Genre co MAE cao nhat (du doan kem nhat) ==========")
display(genre_error.head(5))

print("\n========== Platform co MAE cao nhat (du doan kem nhat) ==========")
display(platform_error.head(5))

## 5.6c So sánh mức độ hữu ích của Critic Score / User Score

In [ ]:
feat_cols_of_interest = [c for c in X_train_encoded.columns if c in
                         ["Critic_Score","Critic_Count","User_Score_100","User_Count","Has_Critic_Score","Has_User_Score"]]

rf_tuned_importance_full = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": rf_model_tuned.feature_importances_
}).set_index("Feature")

print("Muc do quan trong (Random Forest Tuned) cua cac dac trung tu Critic/User Score:")
display(rf_tuned_importance_full.loc[feat_cols_of_interest].sort_values("Importance", ascending=False))

print(f"\nTong ty trong cua nhom dac trung Critic/User Score: {rf_tuned_importance_full.loc[feat_cols_of_interest, 'Importance'].sum():.1%}")

## 5.7 Lưu mô hình tốt nhất

In [ ]:
import joblib

# best_model_name / best_model da duoc xac dinh o muc 5.4b

# Luu kem toan bo thong tin tien xu ly can thiet de tai su dung model tren du lieu moi
# (neu chi luu model se khong tai tao duoc input dung cach)
artifact = {
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": list(X_train_encoded.columns),   # thu tu cot sau One-Hot Encoding
    "publisher_mean_map": publisher_mean_map,
    "genre_platform_mean_map": gp_mean_map,
    "franchise_mean_map": franchise_mean_map,
    "developer_mean_map": developer_mean_map,
    "impute_medians": impute_medians.to_dict(),          # trung vi dung de dien khuyet Critic/User Score
    "global_mean_sales": global_mean_sales,
    "smoothing_k": SMOOTHING_K,
    "selected_features": selected_features,
}

joblib.dump(artifact, "best_vgsales_model.joblib")
print(f"Da luu mo hinh tot nhat ({best_model_name}, R2={best_r2:.4f}) cung toan bo pipeline tien xu ly vao 'best_vgsales_model.joblib'")

## 5.8 Kết luận

- So với **Baseline** (dự đoán bằng trung bình toàn cục), cả 4 mô hình học máy đều cải thiện đáng kể MAE/RMSE/R² — xác nhận các đặc trưng đã xây dựng thực sự mang thông tin hữu ích.
- Việc bổ sung `Critic_Score`, `User_Score`, `Rating`, `Developer_Mean_Sales` từ bộ dữ liệu giàu thông tin hơn (mục 5.6c) cho thấy nhóm đặc trưng đánh giá chuyên môn/người dùng đóng góp đáng kể vào mức độ quan trọng của mô hình — xác nhận giả thuyết đặt ra ở đầu notebook.
- **Random Forest** ổn định và chính xác hơn **Decision Tree** trên cả Cross-Validation lẫn Test, đúng như kỳ vọng của một mô hình *ensemble*.
- Sau khi khắc phục rò rỉ dữ liệu (mục 4.5) và áp dụng **Bayesian Smoothing** cho Target Encoding, các chỉ số R² đo được phản ánh **đúng hơn** khả năng dự đoán thực tế trên dữ liệu chưa từng thấy.
- Sau khi tối ưu siêu tham số (`RandomizedSearchCV`, mục 4.9) và thử thêm **Gradient Boosting** (mục 4.10), mô hình tốt nhất được chọn động ở mục 5.4b dựa trên R² thực tế trên Test.
- Phân tích lỗi theo phân khúc (mục 5.6b) cho thấy mô hình dự đoán kém hơn ở một số Genre/Platform có ít dữ liệu lịch sử — cần lưu ý khi áp dụng cho các tựa game thuộc nhóm này.
- **Đề xuất cải thiện thêm:**
  - Thử `GridSearchCV` với không gian tham số hẹp hơn quanh kết quả của `RandomizedSearchCV` để tinh chỉnh sâu hơn.
  - Thử thêm XGBoost/LightGBM và dùng Permutation Importance hoặc SHAP để đánh giá tầm quan trọng đặc trưng khách quan hơn.
  - Áp dụng Out-of-Fold Target Encoding để tránh rò rỉ dữ liệu nhẹ trong chính quá trình Cross-Validation.
  - Với ~40-51% dữ liệu Critic/User Score bị thiếu, có thể thử các phương pháp điền khuyết nâng cao hơn (KNN Imputer, Iterative Imputer) thay vì chỉ dùng trung vị.